## 5.3 Self supervised training using Language Models

In all previous weeks, you have been training with supervision. You always had a training dataset consisting of an input and a target. The model was trained to make prediction closer to the target. In this week, we take a different approach to training. Here we employ what is called as self-supervised training. 

In self-supervised training, the training data exists as a large corpus of text. This could be text derived from a large collection such as Wikipedia, Internet, or Protein sequence data. 

Sizes of data: 
 - Wikipedia: XX tokens
 - Pre-1930 dataset (from Google Ngram): XX tokens
 - Protein sequence
 - Common Crawl (internet): 

Just as reference, an average human reads XX tokens worth of data throughout their life. Is this a lot? 

We saw in the first module that not just text, but we can also tokenise graph data, and images. The total size of visual data present in internet is XX petabytes, giving rise to approximately YY tokens. An average human toddler observes XX tokens of data within four years. 

How do we perform training? There are two modes: 
- Causal Language Modelling: Predicting the probability distribution of next-token given a sequence of tokens. This is the pre-training objective for large language models.  
- Masked Language Modelling: Predicitng the probability distribution of masked-tokens within a sequence given bidirectional context. Most protein Language Models use masked language modeling. 

This week, we will do both. First let us consider some common measures employed to study the loss behaviour. 

### 5.3.1: Loss function 
Commonly, the cross-entropy loss is used to obtain the output error signal which is used for backpropagation. Cross-entropy loss is computed across the entire vocabulary. 
>> Need to check: Is this true or is it sampled for top-k or top-p while computing loss? 

#### Perplexity 
For validating generated sequence, a common method used is perplexity. This is defined as 
\begin{equation*}
    p = ?? 
\end{equation*}

It is a measure of how *surprising* the generated sequence is. The lower the perplexity, the better. Run the following code cells to get an intuition for perplexity

In [ ]:
## DO SOMETHING HERE

### 5.3.2: Training a Language Model for text generation. 

In the src/lm_utils.py file we have defined the Language Model class as it was written in Module 2. We have left out one function which we need to train the language model for text generation. 

#### Text generation

Generating text from a language model is done by sequentially predicting embedding vector for the next token given an input sequence. We do this by employing a causal mask in the attention head. 

In [ ]:
shakespeare_path = "tinyshakespeare.txt"
with open(shakespeare_path, "r") as f:
    text = f.read()
print(f"Length of text: {len(text)} characters")


Now, get all the unique characters to get a vocabulary


In [ ]:
unique_chars = sorted(set(text))
print(f"Unique characters: {''.join(unique_chars)}")
vocab_size = len(unique_chars)  
print(f"Vocabulary size: {vocab_size}")

We tokenise the chacters by the index in the vocabulary 

In [ ]:
encode = {ch: i for i, ch in enumerate(unique_chars)}
decode = {i: ch for i, ch in enumerate(unique_chars)}

sample_text = "Hello world!"
encoded = [encode[ch] for ch in sample_text]
print(f"Encoded: {encoded}")
decoded = ''.join(decode[i] for i in encoded)
print(f"Decoded: {decoded}")


In [ ]:
import torch

data = torch.tensor([encode[ch] for ch in text], dtype=torch.long)
percent_training = 0.9
n = int(percent_training * len(data))
train_data = data[:n]
val_data = data[n:]
print(f"Training data length: {len(train_data)}")
print(f"Validation data length: {len(val_data)}")


In [9]:
# Create batches of data for training using sequential sampling
def get_batch(data, batch_size, block_size):
    # Randomly select starting indices for the batch
    start_indices = torch.randint(0, len(data) - block_size, (batch_size,))
    
    # Create input and target batches
    x = torch.stack([data[i:i+block_size] for i in start_indices])
    y = torch.stack([data[i+1:i+block_size+1] for i in start_indices])
    
    return x, y

In [ ]:
x_batch, y_batch = get_batch(train_data, batch_size=4, block_size=8)
print("Input batch (x):")
print(x_batch)
print("Target batch (y):")
print(y_batch)


In [ ]:
from src.lm_utils import LanguageModel

n_layers = 4 
n_heads = 4
d_input = 64
context_length = 32
batch_size = 16
d_model = 64

tinyLM = LanguageModel(
    n_layers=n_layers,
    n_head=n_heads,
    d_input=d_input,
    d_model=d_model,
    context_length=context_length,
    vocab_size=vocab_size,
    d_ff=d_model*4
)



In [ ]:
import torch 
import torch.nn as nn

sample_idx = torch.tensor([[1, 3, 4, 5]])
logits, attns = tinyLM(sample_idx, causal=True, return_attention_weights=True)
print(logits.shape)  # Should be (batch_size, seq_length, vocab_size)

In [ ]:
generated = tinyLM.generate(sample_idx, max_new_tokens=5)
print("Generated indices:", generated)
generated_text = ''.join(decode[idx.item()] for idx in generated[0])
print("Generated text:", generated_text)

### 5.3.2 Setting up training 

In [ ]:
learning_rate = 1e-3
max_epochs = 1000
optimizer = torch.optim.Adam(tinyLM.parameters(), lr=learning_rate)
loss = nn.CrossEntropyLoss()


In [ ]:
def evaluate(model, data, block_size):
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for i in range(0, len(data) - block_size, block_size):
            x = data[i:i+block_size].unsqueeze(0)  # Add batch dimension
            y = data[i+1:i+block_size+1].unsqueeze(0)
            logits = model(x, causal=True)
            total_loss += loss(logits.view(-1, vocab_size), y.view(-1)).item()
    return total_loss / (len(data) // block_size)

def generate_text(model, start_text="To be or", max_new_tokens=20):
    model.eval()
    input_indices = torch.tensor([[encode[ch] for ch in start_text]], dtype=torch.long)
    generated_indices = model.generate(input_indices, max_new_tokens=max_new_tokens)
    generated_text = ''.join(decode[idx.item()] for idx in generated_indices[0])
    print(generated_text)
    


In [ ]:
for epoch in range(max_epochs):
    tinyLM.train()
    x_batch, y_batch = get_batch(train_data, batch_size, context_length)
    optimizer.zero_grad()
    logits = tinyLM(x_batch, causal=True)
    train_loss = loss(logits.view(-1, vocab_size), y_batch.view(-1))
    train_loss.backward()
    optimizer.step()

    if epoch % 100 == 0:
        val_loss = evaluate(tinyLM, val_data, context_length)
        print(f"Epoch {epoch}, Train Loss: {train_loss.item():.4f}, Val Loss: {val_loss:.4f}")
        generate_text(tinyLM, start_text="Hello", max_new_tokens=20)

In [ ]:
# After training, we can generate some text to see how well the model has learned
prompt = "To be, or not"
prompt_idx = torch.tensor([[encode[ch] for ch in prompt]], dtype=torch.long)
generated = tinyLM.generate(prompt_idx, max_new_tokens=100, top_k=10)
generated_text = ''.join(decode[idx.item()] for idx in generated[0])
print("Generated text:")
print("-" * 40)
print(generated_text)

In [ ]:
def load_pdb_sequences(path, max_len=256, n_sequences=2000):
    """Read protein sequences from a pdb_seqres FASTA file.

    max_len      : keep only sequences with <= this many residues
    n_sequences  : stop after collecting this many (sample, not the whole file)
    Returns a list of unique uppercase sequence strings.
    """
    seqs = []
    seen = set()
    header = None
    with open(path) as f:
        for line in f:
            line = line.rstrip()
            if line.startswith(">"):
                header = line
            elif header is not None:
                seq = line.upper()
                is_protein = "mol:protein" in header
                if is_protein and len(seq) <= max_len and seq not in seen:
                    seen.add(seq)
                    seqs.append(seq)
                    if len(seqs) >= n_sequences:
                        break
                header = None
    return seqs

### Masked Language Modelling on Sequence Data


In [1]:
import torch
import torch.nn as nn

sequences_path = "protein_dataset.txt"
with open(sequences_path, "r") as f:
    sequences = f.read()


print(f"Length of sequences: {len(sequences)} characters")


Length of sequences: 5322278 characters


In [2]:
# Create a vocabulary of unique characters in the sequences
unique_chars = sorted(set(sequences))
print(f"Unique characters: {''.join(unique_chars)}")
vocab_size = len(unique_chars)
print(f"Vocabulary size: {vocab_size}")
# Create a mask token 
mask_token = "#"
# Add the mask token to the vocabulary if it's not already present
if mask_token not in unique_chars:
    unique_chars.append(mask_token)
    vocab_size += 1
    print(f"Added mask token '{mask_token}' to vocabulary. New vocab size: {vocab_size}")

Unique characters: 
$ABCDEFGHIKLMNPQRSTUVWXYZ^
Vocabulary size: 27
Added mask token '#' to vocabulary. New vocab size: 28


In [3]:
encode = {ch: i for i, ch in enumerate(unique_chars)}
decode = {i: ch for i, ch in enumerate(unique_chars)}

# Encode a sample sequence
sample_sequence = "MKTIIALSYIFCLVFADYKDDDDK"
encoded = [encode[ch] for ch in sample_sequence]
print(f"Encoded: {encoded}")
# Decode the encoded sequence
decoded = ''.join(decode[i] for i in encoded)
print(f"Decoded: {decoded}")

Encoded: [13, 11, 19, 10, 10, 2, 12, 18, 24, 10, 7, 4, 12, 21, 7, 2, 5, 24, 11, 5, 5, 5, 5, 11]
Decoded: MKTIIALSYIFCLVFADYKDDDDK


In [4]:
plm_data = torch.tensor([encode[ch] for ch in sequences], dtype=torch.long)
percent_training = 0.9
n = int(percent_training * len(plm_data))
train_data = plm_data[:n]
val_data = plm_data[n:]
print(f"Training data length: {len(train_data)}")
print(f"Validation data length: {len(val_data)}")


Training data length: 4790050
Validation data length: 532228


In [5]:
from src.lm_utils import LanguageModel

n_layers_plm = 12
n_heads_plm = 8
d_input_plm = 128
d_model_plm = 128
context_length_plm = 32
proteinLM = LanguageModel(
    n_layers=n_layers_plm,
    n_head=n_heads_plm,
    d_input=d_input_plm,
    d_model=d_model_plm,
    context_length=context_length_plm,
    vocab_size=vocab_size,
    d_ff=d_model_plm*4
)


In [6]:
def evaluate_mlm_accuracy(model, data, batch_size, block_size, n_batches=50, mask_ratio=0.15):
    """Fraction of masked tokens the model predicts correctly (validation)."""
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for _ in range(n_batches):
            x_batch, y_batch = get_batch_mlm(data, batch_size, block_size, mask_ratio=mask_ratio)
            logits = model(x_batch, causal=False)          # (B, T, vocab)
            preds = logits.argmax(dim=-1)                  # (B, T) most-likely token

            masked = y_batch != -100                       # only the masked positions count
            correct += (preds[masked] == y_batch[masked]).sum().item()
            total   += masked.sum().item()
    model.train()
    return correct / max(total, 1) * 100.0

# Create batches of data for training using sequential sampling
def get_batch_mlm(data, batch_size, block_size, mask_ratio=0.15):
    # Randomly select starting indices for the batch
    start_indices = torch.randint(0, len(data) - block_size, (batch_size,))
    
    # Create input and target batches
    x = torch.stack([data[i:i+block_size] for i in start_indices])
    
    # special-token IDs that must never be masked or predicted
    special_ids = torch.tensor([encode["^"], encode["$"], encode["\n"]])
    is_special = torch.isin(x, special_ids)

    # choose positions to mask: random, but never a special token
    mask = (torch.rand(x.shape) < mask_ratio) & (~is_special)

    x = x.masked_fill(mask, encode['#'])  # Replace masked positions with 'X' token
    y = torch.stack([data[i:i+block_size] for i in start_indices])
    y = y.masked_fill(~mask, -100)  # Only compute loss on masked positions

    
    return x, y

In [7]:
learning_rate_plm = 1e-4
max_epochs_plm = 1000
optimizer_plm = torch.optim.Adam(proteinLM.parameters(), lr=learning_rate_plm)
loss_plm = nn.CrossEntropyLoss(ignore_index=-100)
batch_size = 16

In [8]:
for epoch in range(max_epochs_plm):
    proteinLM.train()
    x_batch, y_batch = get_batch_mlm(train_data, batch_size, context_length_plm, 
                                     mask_ratio=0.5)
    optimizer_plm.zero_grad()
    logits = proteinLM(x_batch, causal=False)
    train_loss = loss_plm(logits.view(-1, vocab_size), y_batch.view(-1))
    train_loss.backward()
    optimizer_plm.step()
    
    if epoch % 100 == 0:
        acc = evaluate_mlm_accuracy(proteinLM, val_data, batch_size, context_length_plm, mask_ratio=0.5)
        print(f"Epoch {epoch}, Train Loss: {train_loss.item():.4f}, Val Acc: {acc:.1f}%")
        

Epoch 0, Train Loss: 3.5632, Val Acc: 4.7%
Epoch 100, Train Loss: 2.9999, Val Acc: 8.0%
Epoch 200, Train Loss: 2.9491, Val Acc: 8.5%
Epoch 300, Train Loss: 2.8408, Val Acc: 8.1%
Epoch 400, Train Loss: 2.8829, Val Acc: 8.5%
Epoch 500, Train Loss: 2.9210, Val Acc: 8.7%
Epoch 600, Train Loss: 2.8701, Val Acc: 9.8%
Epoch 700, Train Loss: 2.9603, Val Acc: 8.1%
Epoch 800, Train Loss: 2.9214, Val Acc: 9.1%
Epoch 900, Train Loss: 2.9549, Val Acc: 9.6%


### Assignment 3 ?? 

- Using PEPMLM to generate potential binders given a sequence
- Using ESM-2 to predict binding sites
- 